# CellTypist testing

In [ ]:
%matplotlib inline

from pathlib import Path

import scanpy as sc
import celltypist
from celltypist import models

from shared.repo import REPO_ROOT

In [ ]:
data_path = REPO_ROOT / "data/scbasecount/2026-01-12/h5ad/GeneFull/Homo_sapiens/SRX17412841.h5ad"
accession = data_path.stem

adata_raw = sc.read(data_path)

In [ ]:
# Log1p normalize
adata = adata_raw.copy()
sc.pp.normalize_total(adata, target_sum=10_000)
sc.pp.log1p(adata.X)

In [ ]:
adata.X.expm1().sum(axis=1)

In [ ]:
# Download models and show where they are stored
# models.download_models(force_update=True)
models.models_path

In [ ]:
import pandas as pd

d = models.models_description()
d = pd.DataFrame(d[d["model"].str.contains("Lung")])
d

In [ ]:
model = models.Model.load(model="Nuclei_Lung_Airway.pkl")
model.cell_types

In [ ]:
from metadata import MetadataConfig, sample_row_for_srx

from shared.repo import REPO_ROOT

cfg = MetadataConfig(
    sampleParquetPath=REPO_ROOT / "data/scbasecount/2026-01-12/metadata/GeneFull/Homo_sapiens/scbasecount_2026-01-12_metadata_GeneFull_Homo_sapiens_sample_metadata.parquet",
    obsParquetPath=REPO_ROOT / "data/scbasecount/2026-01-12/metadata/GeneFull/Homo_sapiens/scbasecount_2026-01-12_metadata_GeneFull_Homo_sapiens_obs_metadata.parquet",
)

row = sample_row_for_srx(accession, cfg)
row["disease"] if row is not None else None

In [ ]:
adata.var["ensembl_id"] = adata.var_names  # optional: keep Ensembl IDs
adata.var_names = adata.var["gene_symbols"]
adata.var_names_make_unique()

# Generate embeddings
sc.pp.neighbors(adata)

# Annotate
predictions = celltypist.annotate(
    adata,
    model,
    # No clustering step--we are leiden clustering in our method
    majority_voting=False,
)

In [ ]:
adata_pred = predictions.to_adata()

In [ ]:
sc.tl.umap(adata_pred)

In [ ]:
if "majority_voting" in adata_pred.obs.columns:
    color = ["cell_type", "predicted_labels", "majority_voting"]
else:
    color = ["cell_type", "predicted_labels"]

sc.pl.umap(adata_pred, color=color, wspace=0.4, return_fig=True)

## Cluster validation: `cell_type` vs `predicted_labels`

Compare resolution selection and merged Leiden partitions when the weak prior is author `cell_type` labels versus per-cell CellTypist `predicted_labels`.

In [ ]:
%matplotlib inline

import pandas as pd
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

from cluster_validation import ClusterValidationConfig, run_cluster_validation_on_adata
from shared.repo import REPO_ROOT

SRX = "SRX12366723"
OUTPUT_ROOT = REPO_ROOT / "tmp" / "celltypist_cluster_comparison"

base_cfg = ClusterValidationConfig(
    outputDir=OUTPUT_ROOT / "data",
    figsDir=OUTPUT_ROOT / "figs",
)

In [ ]:
# Cluster on raw counts; attach CellTypist labels from adata_pred
adata_for_celltypist_prior = adata_raw.copy()
adata_for_celltypist_prior.obs["predicted_labels"] = adata_pred.obs["predicted_labels"].astype(str)

cfg_cell_type = base_cfg.model_copy(
    update={
        "weakPriorKey": "cell_type",
        "runLabel": f"{SRX}_cell_type",
    }
)
cfg_predicted = base_cfg.model_copy(
    update={
        "weakPriorKey": "predicted_labels",
        "runLabel": f"{SRX}_predicted_labels",
    }
)

adata_cell_type, result_cell_type = run_cluster_validation_on_adata(
    adata_raw.copy(),
    cfg_cell_type,
    SRX,
)
adata_predicted, result_predicted = run_cluster_validation_on_adata(
    adata_for_celltypist_prior,
    cfg_predicted,
    SRX,
)

In [ ]:
def _best_jacc(result) -> float:
    return float(result.jaccArr[result.resolutions.index(result.selectedResolution)])


comparison = pd.DataFrame(
    [
        {
            "run": result_cell_type.runTag,
            "weak_prior": result_cell_type.weakPriorKey,
            "selected_resolution": result_cell_type.selectedResolution,
            "k_prior": result_cell_type.kPrior,
            "k_filtered": result_cell_type.kFiltered,
            "n_cells_final": result_cell_type.nCellsFinal,
            "n_clusters_pre_merge": result_cell_type.nClustersPreMerge,
            "n_clusters_post_merge": result_cell_type.nClustersPostMerge,
            "best_jaccard": _best_jacc(result_cell_type),
        },
        {
            "run": result_predicted.runTag,
            "weak_prior": result_predicted.weakPriorKey,
            "selected_resolution": result_predicted.selectedResolution,
            "k_prior": result_predicted.kPrior,
            "k_filtered": result_predicted.kFiltered,
            "n_cells_final": result_predicted.nCellsFinal,
            "n_clusters_pre_merge": result_predicted.nClustersPreMerge,
            "n_clusters_post_merge": result_predicted.nClustersPostMerge,
            "best_jaccard": _best_jacc(result_predicted),
        },
    ]
).set_index("run")

shared_cells = adata_cell_type.obs_names.intersection(adata_predicted.obs_names)
labels_a = adata_cell_type.obs.loc[shared_cells, "leiden_merged"]
labels_b = adata_predicted.obs.loc[shared_cells, "leiden_merged"]

cross_run = pd.Series(
    {
        "n_cells_shared": len(shared_cells),
        "ari_vs_other_run": adjusted_rand_score(labels_a, labels_b),
        "nmi_vs_other_run": normalized_mutual_info_score(labels_a, labels_b),
    },
    name="cross_run",
)

comparison, cross_run

In [ ]:
fig_cell_type = sc.pl.umap(
    adata_cell_type,
    color=["leiden_merged", "cell_type"],
    title=[f"cell_type prior ({result_cell_type.nClustersPostMerge} clusters)", "cell_type"],
    wspace=0.4,
    show=False,
    return_fig=True,
)
fig_predicted = sc.pl.umap(
    adata_predicted,
    color=["leiden_merged", "predicted_labels"],
    title=[f"predicted_labels prior ({result_predicted.nClustersPostMerge} clusters)", "predicted_labels"],
    wspace=0.4,
    show=False,
    return_fig=True,
)
fig_cell_type, fig_predicted